autotunenet comparison 1 of 2

Generated by `autotunenet prepare --runtime kaggle`. Runs part of the
AutoTuneNet benchmark experiment `comparison` on Kaggle's GPU T4 x2:
one group of cells per GPU, in parallel.

- installs: `autotunenet[bench] @ git+https://github.com/mitishraina/AutoTuneNet.git@2c07032f1725a75764d9fa0cb88587a8cfe4d3ac`
- cells: 11 (6 + 5 per GPU)

Steps: install, write the experiment (and any plugins), link the attached
datasets, run, and pack the results into `autotunenet_results.tar.gz`.

In [ ]:
import glob
import json
import os
import subprocess
import tarfile
import time

JOB = json.loads(r'''{"install": "autotunenet[bench] @ git+https://github.com/mitishraina/AutoTuneNet.git@2c07032f1725a75764d9fa0cb88587a8cfe4d3ac", "experiment": "comparison", "experiment_yaml": "# AutoTuneNet vs. other learning-rate strategies.\n#   autotunenet run --experiment comparison\nname: comparison\nepochs: 10\nseeds: [42, 43, 44]\ntasks: [mnist, cifar10]\n\nscenarios:\n  stationary: {}\n  label_noise: {start_epoch: 5, noise_ratio: 0.3}            # 30% of training labels randomised\n  gradient_noise: {start_epoch: 5, std: 0.02, tasks: [mnist]} # noise added to every gradient\n  # Stress test. Offline search methods have no starting learning rate (each\n  # trial samples its own), so it does not apply to them.\n  bad_initial_lr:\n    initial_lr: 0.1\n    methods: [fixed_lr, cosine, plateau, autotunenet, pbt]   # PBT members all start from the bad LR\n\n# Each method's keyword arguments. All tuning methods search the task's lr_range on a log scale.\nmethods:\n  fixed_lr: {}\n  cosine: {}\n  plateau: {factor: 0.5, patience: 1}                       # halve LR after 2 epochs in a row without improvement\n  autotunenet:\n    n_startup_trials: 3\n    warmup_epochs: 2\n    max_delta: 0.5\n    smoothing_window: 3\n    max_regression: 0.2\n    patience: 2\n    cooldown: 3\n  # Offline search: separate trials, each a full run with its own LR; best on validation is reported.\n  optuna_tpe: {n_trials: 8, n_startup_trials: 3}           # 8 full trials\n  asha: {n_trials: 16, n_startup_trials: 3}                # stops weak trials early\n  hyperband: {n_trials: 16, n_startup_trials: 3}\n  # Population-based training (Ray Tune): 4 models in parallel; every 2 epochs the weakest\n  # copy the best and perturb the LR (x1.2 / x0.8, or resample).\n  pbt: {population: 4, perturbation_interval: 2}\n", "plugins": {}, "gpu_cells": [[{"task": "cifar10", "scenario": "bad_initial_lr", "method": "pbt", "seed": 42}, {"task": "cifar10", "scenario": "label_noise", "method": "pbt", "seed": 43}, {"task": "cifar10", "scenario": "stationary", "method": "pbt", "seed": 44}, {"task": "mnist", "scenario": "gradient_noise", "method": "pbt", "seed": 42}, {"task": "mnist", "scenario": "label_noise", "method": "pbt", "seed": 43}, {"task": "mnist", "scenario": "stationary", "method": "pbt", "seed": 44}], [{"task": "cifar10", "scenario": "bad_initial_lr", "method": "pbt", "seed": 43}, {"task": "cifar10", "scenario": "label_noise", "method": "pbt", "seed": 44}, {"task": "mnist", "scenario": "bad_initial_lr", "method": "pbt", "seed": 42}, {"task": "mnist", "scenario": "gradient_noise", "method": "pbt", "seed": 43}, {"task": "mnist", "scenario": "label_noise", "method": "pbt", "seed": 44}]]}''')
WORK = "/kaggle/temp/autotunenet"          # outside /kaggle/working, so not part of the output
DATA = os.path.join(WORK, "data")
RESULTS = "/kaggle/working/results"
LOGS = "/kaggle/working/logs"
for folder in (WORK, DATA, LOGS):
    os.makedirs(folder, exist_ok=True)
START = time.time()


def sh(cmd, **kwargs):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, **kwargs)

## 1. Install AutoTuneNet and write the experiment

In [ ]:
sh(f"pip install -q '{JOB['install']}'")
experiment = os.path.join(WORK, "experiment.yaml")
open(experiment, "w").write(JOB["experiment_yaml"])
plugins = []
for name, source in JOB["plugins"].items():
    path = os.path.join(WORK, name)
    open(path, "w").write(source)
    plugins += ["--plugins", path]
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")

## 2. Data
Each attached `autotunenet-<task>` dataset holds the files its task reads, relative to the data
directory. Kaggle may keep the uploaded archive or unpack it; both are handled.

In [ ]:
for archive in glob.glob("/kaggle/input/**/*.tar", recursive=True):
    with tarfile.open(archive) as tar:
        tar.extractall(DATA, filter="data")
for marker in glob.glob("/kaggle/input/**/autotunenet_data.json", recursive=True):
    root = os.path.dirname(marker)
    for entry in os.listdir(root):
        target = os.path.join(DATA, entry)
        if entry != "autotunenet_data.json" and not os.path.exists(target):
            os.symlink(os.path.join(root, entry), target)
print("data:", sorted(os.listdir(DATA)))
ENV = {**os.environ, "AUTOTUNENET_DATA_DIR": DATA}

## 3. Run one group of cells per GPU

In [ ]:
procs = []
for gpu, cells in enumerate(JOB["gpu_cells"]):
    if not cells:
        continue
    cells_file = os.path.join(WORK, f"cells_gpu{gpu}.json")
    json.dump(cells, open(cells_file, "w"))
    cmd = ["autotunenet", "run", "--experiment", experiment, "--cells", cells_file,
           "--results-dir", RESULTS, "--device", "cuda", "--no-report"] + plugins
    log_path = os.path.join(LOGS, f"gpu{gpu}.log")
    print(f"GPU {gpu}: {len(cells)} cells", flush=True)
    procs.append((gpu, log_path, subprocess.Popen(
        cmd, cwd=WORK, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)},
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT)))
failed = [gpu for gpu, _, p in procs if p.wait() != 0]
print(f"finished in {(time.time() - START) / 60:.1f} min; GPUs with failed cells: {failed or 'none'}")

## 4. Package the results

In [ ]:
with tarfile.open("/kaggle/working/autotunenet_results.tar.gz", "w:gz") as tar:
    if os.path.isdir(RESULTS):
        tar.add(RESULTS, arcname="results")
    tar.add(LOGS, arcname="logs")
for gpu, log_path, _ in procs:
    print(f"--- last lines of GPU {gpu} ---")
    print("".join(open(log_path).readlines()[-15:]))